# 原 Stage1 09:46 配置的同日同业早盘广度来源
仅取 AMD/NVDA/SMH 三根已完成 5min bar，同涨比例；来源、时点和覆盖预检。0 标签、0 拟合、0 账户。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_0946_peer_breadth_source_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
s=plan['source'];assert sha(root/s['provider_uri']/'qlib_store_manifest.json')==s['provider_manifest_sha256'] and sha(root/s['parent_card']/'summary.json')==s['parent_summary_sha256'] and sha(root/s['prior_1031_source_card']/'plan.json')==s['prior_1031_plan_sha256']
parent=json.loads((root/s['parent_card']/'summary.json').read_text());assert parent['recorder_id']==s['parent_recorder_id'] and parent['nas_evidence']['sha256']==s['parent_NAS_archive_sha256']
readback=json.loads((root/s['parent_NAS_readback']).read_text());assert readback['NAS_archive_sha256']==s['parent_NAS_archive_sha256']
prior_plan=json.loads((root/s['prior_1031_source_card']/'plan.json').read_text());assert prior_plan['source']['parent_NAS_archive_sha256']==s['parent_NAS_archive_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/s['provider_uri']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2018-10-15 09:35',end_time='2026-10-08 09:45',freq='5min'));days=pd.DatetimeIndex(calendar.normalize().unique())
cols=['$open','$close','$price_valid'];raw=D.features(plan['feature']['instruments'],cols,start_time='2018-10-15 09:35',end_time='2026-10-08 09:45',freq='5min')
assert isinstance(raw.index,pd.MultiIndex) and set(raw.index.names)=={'instrument','datetime'} and raw.index.is_unique
bars=raw.reset_index();bars['datetime']=pd.to_datetime(bars['datetime']);bars['trade_date']=bars.datetime.dt.normalize();bars['clock']=bars.datetime.dt.strftime('%H:%M');clocks=plan['feature']['required_right_labelled_bars'];early=bars.loc[bars.clock.isin(clocks)].copy()
groups={(d,inst):g.sort_values('datetime') for (d,inst),g in early.groupby(['trade_date','instrument'],sort=False)}
rows=[];details=[]
for day in days:
 positive=[];status='known';invalid_reason=None
 for inst in plan['feature']['instruments']:
  g=groups.get((day,inst),early.iloc[0:0]);valid=len(g)==3 and g.clock.tolist()==clocks and g['$open'].notna().all() and g['$close'].notna().all() and np.isfinite(g[['$open','$close','$price_valid']].to_numpy(float)).all() and (g[['$open','$close']].to_numpy(float)>0).all() and g['$price_valid'].eq(1).all()
  if not valid:
   status='unknown';invalid_reason='missing_or_invalid_'+inst;details.append({'trade_date':day,'instrument':inst,'bars':len(g),'valid':False,'log_opening_15m_return':np.nan});continue
  ret=float(np.log(float(g.iloc[-1]['$close'])/float(g.iloc[0]['$open'])));positive.append(ret>0);details.append({'trade_date':day,'instrument':inst,'bars':3,'valid':True,'log_opening_15m_return':ret})
 fraction=float(np.mean(positive)) if status=='known' and len(positive)==3 else np.nan
 rows.append({'trade_date':day,'peer_positive_fraction_3':fraction,'source_known':bool(np.isfinite(fraction)),'status':status,'invalid_reason':invalid_reason,'assumed_available_at':day+pd.Timedelta(hours=9,minutes=45),'receiver_decision_at':day+pd.Timedelta(hours=9,minutes=46)})
daily=pd.DataFrame(rows);detail=pd.DataFrame(details);assert len(daily)==len(days) and daily.trade_date.equals(pd.Series(days,name='trade_date')) and detail.groupby('trade_date').size().eq(3).all()
train=daily.loc[daily.trade_date.between(*[pd.Timestamp(x) for x in plan['calendar']['train_action']])];held=daily.loc[daily.trade_date.between(*[pd.Timestamp(x) for x in plan['calendar']['reused_held_action']])]
audit={'status':'opening_peer_breadth_source_gate_passed' if len(train)==plan['calendar']['expected_train_days'] and len(held)==plan['calendar']['expected_held_days'] and train.source_known.mean()>=plan['calendar']['minimum_known_fraction_each'] and held.source_known.mean()>=plan['calendar']['minimum_known_fraction_each'] else 'opening_peer_breadth_source_gate_failed','plan_sha256':sha(card/'plan.json'),'source_calendar_days':len(days),'training_calendar_days':len(train),'training_known_days':int(train.source_known.sum()),'held_calendar_days':len(held),'held_known_days':int(held.source_known.sum()),'training_known_fraction':float(train.source_known.mean()),'held_known_fraction':float(held.source_known.mean()),'unknown_train_dates':[str(x.date()) for x in train.loc[~train.source_known,'trade_date']],'unknown_held_dates':[str(x.date()) for x in held.loc[~held.source_known,'trade_date']],'feature_value_counts':{str(k):int(v) for k,v in daily.peer_positive_fraction_3.value_counts(dropna=False).items()},'first_receipt_finality_qualified':False,'market_outcome_label_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n');assert audit['status']=='opening_peer_breadth_source_gate_passed'
assert (daily.assumed_available_at<daily.receiver_decision_at).all()
static=pd.concat({'feature':daily.set_index('trade_date')[['peer_positive_fraction_3']].rename(columns={'peer_positive_fraction_3':plan['feature']['name']}),'meta':daily.set_index('trade_date')[['source_known','assumed_available_at','receiver_decision_at']]},axis=1);static.index=pd.MultiIndex.from_arrays([daily.trade_date,['SOXL_SOXX_CASH']*len(daily)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['calendar']['train_action']),'held':tuple(plan['calendar']['reused_held_action'])});tr=dataset.prepare('train',col_set=['feature','meta']);he=dataset.prepare('held',col_set=['feature','meta']);assert len(tr)==len(train) and len(he)==len(held)
state={'status':'started','plan_sha256':sha(card/'plan.json'),'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset,source_audit=audit);rec.save_objects(artifact_path='analysis',daily=daily,source_opening_15m=detail)
 rec.log_metrics(training_known=int(train.source_known.sum()),held_known=int(held.source_known.sum()),model_fits=0,native_accounts=0)
 with R.uri_context(str(tracking)):reread=R.get_recorder(recorder_id=rec.id,experiment_id=str(rec.experiment_id)).load_object('analysis/daily')
 assert reread.equals(daily)
summary={'experiment':card.name,'decision':'conditional_0946_peer_breadth_source_gate_passed_first_receipt_unqualified_no_fit','objective_achieved':False,'source_audit':'source_audit.json','source_calendar_days':len(days),'training_calendar_days':len(train),'training_known_days':int(train.source_known.sum()),'held_calendar_days':len(held),'held_known_days':int(held.source_known.sum()),'unknown_training_dates':audit['unknown_train_dates'],'unknown_held_dates':audit['unknown_held_dates'],'first_receipt_finality_qualified':False,'new_model_fits':0,'new_native_accounts':0,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/source_opening_15m']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':card.name,'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='opening-peer-breadth-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  names=[x.name for x in source.getmembers() if x.isfile()];assert any(x.endswith('/artifacts/analysis/daily') for x in names) and any(x.endswith('/artifacts/inputs/dataset') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['NAS_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'training_known':summary['training_known_days'],'held_known':summary['held_known_days'],'recorder_id':state['recorder_id'],'NAS_sha256':evidence['sha256']})


{'decision': 'conditional_0946_peer_breadth_source_gate_passed_first_receipt_unqualified_no_fit', 'training_known': 1061, 'held_known': 944, 'recorder_id': 'e7312100d7274be7aca700754c900f0a', 'NAS_sha256': '3947e13dfd1decd685992063a2d13cad3144cef2b9ec831df6c9cdcc5e373120'}


## 中文结论
# Stage1 09:46 同业早盘广度来源预检

冻结 AMD、NVDA、SMH 09:35、09:40、09:45 三根已完成的 5 分钟 bar；每只标的取 09:45 收盘相对 09:35 开盘的收益是否为正，再取三只的同涨比例。Qlib 来源核验通过，2018–2022 原 Stage1 训练期 1061/1061 日、2023–2026 复用历史期 944/944 日已知。

原始 Stage1 09:46 配置模型接收的是另一组 39 项输入；本卡只建立新坐标的时钟和覆盖，不读取未来收益、不拟合、不跑账户。09:45 完成 bar 在 09:46 决策前可用仅是封存 bar 时钟假设，历史首收与最终版本未获核证，后续信息检验只能是条件性开发证据。

原生 Qlib Recorder `e7312100d7274be7aca700754c900f0a` 已读回；Yanxuan NAS 归档 SHA-256 `3947e13dfd1decd685992063a2d13cad3144cef2b9ec831df6c9cdcc5e373120` 已远端回读。结论：`conditional_0946_peer_breadth_source_gate_passed_first_receipt_unqualified_no_fit`。下一步另行冻结原 39 项对照加一个坐标的 SOXL−SOXX H5 信息检验；即使通过，仍须来源资格、完整 Stage1 0bp Tiered 独立资金账户及 v2 其他门槛。
